In [40]:
import importlib
import os
import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.fft import fft, fftfreq
from pathlib import Path
from utils import data_loader_utils

importlib.reload(data_loader_utils)

<module 'utils.data_loader_utils' from '/home/scopellie/EECE5644_Project/utils/data_loader_utils.py'>

### Initial data visualization / validation of format

In [38]:
# Path for main dataset data folder
data_path = Path("./data").absolute()
# List of all h5 files in the data folder
file_list = data_loader_utils.find_all_h5s_in_dir(data_path)
print(len(file_list))
print(file_list[0])

1702
/home/scopellie/EECE5644_Project/data/M01/OP02/good/M01_Aug_2019_OP02_013.h5


In [39]:
# Test - Reading and formatting first file
file_df = data_loader_utils.format_file(file_list[0])
file_df

,X,Y,Z,timestamp,label,machine,process,i,window
0,-15.0,25.0,-1013.0,0.0,True,1,2,13,0
1,-21.0,19.0,-1013.0,0.5,True,1,2,13,0
2,-17.0,29.0,-1013.0,1.0,True,1,2,13,0
3,-13.0,13.0,-1013.0,1.5,True,1,2,13,0
4,-9.0,25.0,-1020.0,2.0,True,1,2,13,0
...,...,...,...,...,...,...,...,...,...
94203,0.0,23.0,-1013.0,47101.5,True,1,2,13,23
94204,-19.0,11.0,-1018.0,47102.0,True,1,2,13,23
94205,-1.0,31.0,-1015.0,47102.5,True,1,2,13,23
94206,-19.0,9.0,-1020.0,47103.0,True,1,2,13,23


### Extraction of data window features

In [ ]:
window_size = 4000
windows = []
df = pd.DataFrame()

for file in file_list[:]:
    with h5py.File(file, 'r') as f:
        dataset = f['vibration_data'][:]
        num_windows = len(dataset) // window_size
        for i in range(0, num_windows):
            w = dataset[i * window_size: (i + 1) * window_size]
            features = data_loader_utils.extract_window_features(w.transpose(), file.split("/")[-2] + "/" + file.split("/")[-1])
            df_feat = pd.DataFrame([features])
            df = pd.concat([df, df_feat], ignore_index=True)

print(df.shape)
df.head(5)

/home/scopellie/EECE5644_Project/utils/data_loader_utils.py:184: UserWarning: nperseg=1024 is greater than signal length max(len(x), len(y)) = 3, using nperseg = 3
  f, psd = signal.welch(x, fs=fs, nperseg=nperseg, window="hann")


(43847, 101)


,x_mean,x_std,x_min,x_max,x_ptp,x_rms,x_abs_mean,x_skew,x_kurtosis,x_crest_factor,...,mag_spectral_centroid,mag_spectral_spread,mag_spectral_skew,mag_spectral_kurtosis,mag_spectral_entropy,mag_spectral_rolloff_85,mag_band_power_low,mag_band_power_mid,mag_band_power_high,filename
0,-334.333333,588.082761,-1013.0,25.0,1038.0,585.097997,351.000000,-0.703428,-1.5,1.731334,...,644.105283,120.548358,-5.155972,27.584042,0.213308,666.666667,0.0,0.0,0.0,good/M01_Aug_2019_OP02_013.h5
1,-275.333333,659.454573,-1028.0,201.0,1229.0,604.755047,410.000000,-0.634643,-1.5,1.699862,...,646.949562,112.942340,-5.553564,31.842068,0.192259,666.666667,0.0,0.0,0.0,good/M01_Aug_2019_OP02_013.h5
2,-218.000000,635.925310,-927.0,302.0,1229.0,563.138231,419.333333,-0.497771,-1.5,1.646132,...,659.424310,69.107060,-9.437270,90.062056,0.086465,666.666667,0.0,0.0,0.0,good/M01_Aug_2019_OP02_013.h5
3,-429.333333,697.947228,-1235.0,-9.0,1226.0,713.498890,429.333333,-0.705107,-1.5,1.730907,...,648.128426,109.613690,-5.743719,33.990303,0.183274,666.666667,0.0,0.0,0.0,good/M01_Aug_2019_OP02_013.h5
4,-276.333333,640.944095,-1007.0,191.0,1198.0,591.804866,403.666667,-0.627377,-1.5,1.701574,...,649.141422,106.660032,-5.921770,36.067360,0.175420,666.666667,0.0,0.0,0.0,good/M01_Aug_2019_OP02_013.h5


### Splitting training and testing buckets

In [42]:
good_file_windows = df[df["filename"].str.contains("good")]
bad_file_windows = df[df["filename"].str.contains("bad")]

good_windows = good_file_windows.shape[0]
bad_windows = bad_file_windows.shape[0]
print(f"Windows with good label: {good_windows}; Windows with bad label: {bad_windows}.")

Windows with good label: 42472; Windows with bad label: 1375.


In [43]:
train_ratio = 0.8 + 0.8 * (bad_windows / good_windows)
test_ratio = 1 - train_ratio
n_train = round(good_windows * train_ratio,0)
n_test_good = round(good_windows * test_ratio, 0)

print(f"""Training ratio: {n_train / (good_windows + bad_windows)}; 
{n_train} training vectors \n""")
print(f"""Testing ratio: {(n_test_good + bad_windows) / (good_windows + bad_windows)};
{n_test_good + bad_windows} testing vectors, {n_test_good} with "good" label""")

Training ratio: 0.8000091226309668; 
35078.0 training vectors 

Testing ratio: 0.19999087736903323;
8769.0 testing vectors, 7394.0 with "good" label


In [44]:
df_train = good_file_windows.sample(frac=train_ratio, random_state=42)
df_test = df.drop(df_train.index)

### Model training